In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import average_precision_score

import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
df = pd.read_csv("creditcard.csv")

In [3]:
print(df.shape)

print(df.head())

(284807, 31)
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26       V

In [4]:
print(df["Class"].value_counts())

Class
0    284315
1       492
Name: count, dtype: int64


In [5]:
X = df.drop("Class", axis=1)

y = df["Class"]

In [6]:
scaler = StandardScaler()

X["Amount"] = scaler.fit_transform(
    X[["Amount"]]
)

X["Time"] = scaler.fit_transform(
    X[["Time"]]
)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [8]:
X_train = torch.tensor(
    X_train.values,
    dtype=torch.float32
)

X_test = torch.tensor(
    X_test.values,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train.values,
    dtype=torch.float32
).reshape(-1,1)

y_test = torch.tensor(
    y_test.values,
    dtype=torch.float32
).reshape(-1,1)

In [9]:
frauds = y_train.sum()

normal = len(y_train) - frauds

pos_weight = torch.tensor(
    [normal / frauds]
)

print(pos_weight)

tensor([577.2868])


In [10]:
class LogisticRegressionModel(nn.Module):

    def __init__(self,input_size):

        super().__init__()

        self.linear = nn.Linear(
            input_size,
            1
        )

    def forward(self,x):

        return self.linear(x)

In [22]:
with torch.no_grad():

    logits = linear_model(X_test)

    probs = torch.sigmoid(logits)

    preds = (probs >= 0.5).float()

precision = precision_score(y_test, preds)
recall = recall_score(y_test, preds)
f1 = f1_score(y_test, preds)
prauc = average_precision_score(y_test, probs)

print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("PR-AUC:", prauc)

# Save values
lr_precision = precision
lr_recall = recall
lr_f1 = f1
lr_prauc = prauc

Precision: 0.0025950390940954436
Recall: 0.7857142857142857
F1: 0.00517299294591871
PR-AUC: 0.290311630744264


In [11]:
linear_model = LogisticRegressionModel(
    X_train.shape[1]
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

optimizer = optim.Adam(
    linear_model.parameters(),
    lr=0.001
)

In [12]:
epochs = 50

for epoch in range(epochs):

    outputs = linear_model(X_train)

    loss = criterion(
        outputs,
        y_train
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if epoch % 5 == 0:

        print(
            f"Epoch {epoch}, Loss={loss.item():.4f}"
        )

Epoch 0, Loss=3.6561
Epoch 5, Loss=3.3435
Epoch 10, Loss=3.0424
Epoch 15, Loss=2.7561
Epoch 20, Loss=2.4885
Epoch 25, Loss=2.2428
Epoch 30, Loss=2.0211
Epoch 35, Loss=1.8246
Epoch 40, Loss=1.6537
Epoch 45, Loss=1.5082


In [13]:
with torch.no_grad():

    logits = linear_model(X_test)

    probs = torch.sigmoid(logits)

    preds = (probs >= 0.5).float()


In [14]:
precision = precision_score(
    y_test,
    preds
)

recall = recall_score(
    y_test,
    preds
)

f1 = f1_score(
    y_test,
    preds
)

prauc = average_precision_score(
    y_test,
    probs
)

In [15]:
print("Precision:",precision)
print("Recall:",recall)
print("F1:",f1)
print("PR-AUC:",prauc)

Precision: 0.0025950390940954436
Recall: 0.7857142857142857
F1: 0.00517299294591871
PR-AUC: 0.290311630744264


In [16]:
class FraudMLP(nn.Module):

    def __init__(self,input_size):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_size,64),
            nn.ReLU(),

            nn.Linear(64,32),
            nn.ReLU(),

            nn.Linear(32,16),
            nn.ReLU(),

            nn.Linear(16,1)
        )

    def forward(self,x):

        return self.network(x)

In [17]:
mlp_model = FraudMLP(
    X_train.shape[1]
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

optimizer = optim.Adam(
    mlp_model.parameters(),
    lr=0.001
)

In [18]:
epochs = 50

for epoch in range(epochs):

    outputs = mlp_model(X_train)

    loss = criterion(
        outputs,
        y_train
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if epoch % 5 == 0:

        print(
            f"Epoch {epoch}, Loss={loss.item():.4f}"
        )

Epoch 0, Loss=1.3944
Epoch 5, Loss=1.2442
Epoch 10, Loss=1.1334
Epoch 15, Loss=1.0424
Epoch 20, Loss=0.9618
Epoch 25, Loss=0.8877
Epoch 30, Loss=0.8175
Epoch 35, Loss=0.7508
Epoch 40, Loss=0.6865
Epoch 45, Loss=0.6230


In [19]:
with torch.no_grad():

    logits = mlp_model(X_test)

    probs = torch.sigmoid(logits)

    preds = (probs >= 0.5).float()

In [20]:
mlp_precision = precision
mlp_recall = recall
mlp_f1 = f1
mlp_prauc = prauc

In [23]:
comparison = pd.DataFrame({

    "Model":[
        "Logistic Regression",
        "MLP"
    ],

    "Precision":[
        lr_precision,
        mlp_precision
    ],

    "Recall":[
        lr_recall,
        mlp_recall
    ],

    "F1":[
        lr_f1,
        mlp_f1
    ],

    "PR-AUC":[
        lr_prauc,
        mlp_prauc
    ]
})

print(comparison)

                 Model  Precision    Recall        F1    PR-AUC
0  Logistic Regression   0.002595  0.785714  0.005173  0.290312
1                  MLP   0.002595  0.785714  0.005173  0.290312
